In [6]:
import pandas as pd
import pandas_market_calendars as mcal
from pathlib import Path

DATA_DIR   = Path("Data download/data")
CACHE_FILE = DATA_DIR / "prices_cache.csv"
LABELED    = DATA_DIR / "filings_labeled.csv"

PRICE_START = "2021-01-01"
PRICE_END   = "2026-04-30"

# ── Rebuild NYSE calendar — identical to notebook 03 ─────────────────────────
nyse         = mcal.get_calendar("NYSE")
schedule     = nyse.schedule(start_date=PRICE_START, end_date=PRICE_END)
trading_days = pd.DatetimeIndex(schedule.index.date).normalize()

def prev_trading_day(date: pd.Timestamp) -> pd.Timestamp:
    past = trading_days[trading_days < date]
    return past[-1] if len(past) > 0 else None

print(f"NYSE calendar: {len(trading_days)} trading days")

NYSE calendar: 1337 trading days


In [7]:
# ── Load the price cache (already contains all tickers + SPY, 2021–2026) ─────
raw_prices = pd.read_csv(CACHE_FILE, index_col=0, parse_dates=True)
raw_prices.index = pd.to_datetime(raw_prices.index).normalize()

print(f"Price cache shape : {raw_prices.shape}")
print(f"Cache date range  : {raw_prices.index.min().date()} → {raw_prices.index.max().date()}")
print(f"SPY in cache      : {'SPY' in raw_prices.columns}")

# ── Load filings ──────────────────────────────────────────────────────────────
df = pd.read_csv(LABELED, parse_dates=['event_date', 'event_date_m1'])
print(f"\nFilings loaded    : {len(df)}")
print(f"Columns           : {list(df.columns)}")

Price cache shape : (1059, 529)
Cache date range  : 2021-01-04 → 2026-04-24
SPY in cache      : True

Filings loaded    : 5284
Columns           : ['ticker', 'cik', 'accessionNumber', 'filingDate', 'filingDatetime', 'event_date', 'event_date_m1', 'event_date_p1', 'price_m1', 'price_0', 'price_p1', 'spy_price_m1', 'spy_price_0', 'spy_price_p1', 'mm_alpha', 'mm_beta', 'mm_flag', 'ret_stock', 'ret_spy', 'expected_ret', 'car_0_1', 'car_0_1_mktadj', 'downside']


In [8]:
def get_price(ticker: str, date: pd.Timestamp) -> float:
    """Identical to notebook 03's get_price() function."""
    if ticker not in raw_prices.columns:
        return None
    date = pd.Timestamp(date).normalize()
    if date not in raw_prices.index:
        return None
    val = raw_prices.loc[date, ticker]
    return float(val) if pd.notna(val) else None

# ── Compute event_date_m2 = trading day before event_date_m1 ─────────────────
print("Computing event_date_m2...")
df['event_date_m2'] = df['event_date_m1'].apply(prev_trading_day)

# ── Look up prices on day -2 ──────────────────────────────────────────────────
print("Looking up price_m2 and spy_price_m2...")
df['price_m2']     = df.apply(lambda r: get_price(r['ticker'], r['event_date_m2']), axis=1)
df['spy_price_m2'] = df.apply(lambda r: get_price('SPY',       r['event_date_m2']), axis=1)

# ── Sanity checks ─────────────────────────────────────────────────────────────
missing_stock = df['price_m2'].isna().sum()
missing_spy   = df['spy_price_m2'].isna().sum()
print(f"\nMissing price_m2     : {missing_stock} / {len(df)}")
print(f"Missing spy_price_m2 : {missing_spy} / {len(df)}")

# Gap between m2 and m1 should be 1 calendar day (Mon–Thu) or 3 (Fri→Mon)
df['_gap'] = (df['event_date_m1'] - df['event_date_m2']).dt.days
print(f"\nCalendar days between event_date_m2 and event_date_m1:")
print(df['_gap'].value_counts().sort_index().to_string())
# Expect: 1 (most rows), 3 (Mon events), 4 (after holiday weekends)

Computing event_date_m2...
Looking up price_m2 and spy_price_m2...

Missing price_m2     : 0 / 5284
Missing spy_price_m2 : 0 / 5284

Calendar days between event_date_m2 and event_date_m1:
_gap
1    3947
2      12
3    1191
4     134


In [9]:
# ── Quick verification: ret on day -1 should look like a normal daily return ─
df['_ret_m1_check'] = (df['price_m1'] - df['price_m2']) / df['price_m2']

print("Day -1 stock return distribution (should look like normal daily returns):")
print(df['_ret_m1_check'].describe(percentiles=[.01, .25, .50, .75, .99]).round(4))
# Mean should be near 0, std around 0.015–0.025, no values above ±30%

# Cross-check: NOT the same as ret_stock (day +1 return)
corr = df['_ret_m1_check'].corr(df['ret_stock'])
print(f"\nCorrelation of day -1 return with ret_stock (day +1): {corr:.4f}")
print("(Should be low, ~0.0 to 0.1 — if close to 1.0 the bug is still present)")

Day -1 stock return distribution (should look like normal daily returns):
count    5284.0000
mean        0.0011
std         0.0186
min        -0.1433
1%         -0.0471
25%        -0.0086
50%         0.0012
75%         0.0108
99%         0.0480
max         0.1756
Name: _ret_m1_check, dtype: float64

Correlation of day -1 return with ret_stock (day +1): -0.0009
(Should be low, ~0.0 to 0.1 — if close to 1.0 the bug is still present)


In [10]:
# ── Drop helper columns, save patched CSV ────────────────────────────────────
df.drop(columns=['_gap', '_ret_m1_check'], inplace=True)

df.to_csv(LABELED, index=False)

print(f"Saved {len(df)} filings → {LABELED}")
print(f"New columns added: event_date_m2, price_m2, spy_price_m2")
print(f"\nSample (first 3 rows):")
print(df[['ticker', 'event_date_m2', 'event_date_m1', 'price_m2', 'price_m1']].head(3).to_string())

Saved 5284 filings → Data download/data/filings_labeled.csv
New columns added: event_date_m2, price_m2, spy_price_m2

Sample (first 3 rows):
  ticker event_date_m2 event_date_m1    price_m2    price_m1
0      A    2026-02-23    2026-02-24  123.917244  124.116791
1      A    2025-11-20    2025-11-21  144.471542  150.636444
2      A    2025-08-25    2025-08-26  118.428841  117.583984
